In [1]:
 # Practical Assessment: Intelligent Document Processing + MLOps + AI Search

In [2]:
# Scenario


# You are working for an Insurance Company that wants to:
# Extract data from claim forms (PDF/Images)
# Store and search documents using AI Search
# Train a prediction model (fraud detection / claim approval)
# Deploy and monitor the model using MLOps
# Integrate everything using APIs

In [3]:
# PART 1

In [10]:
from dotenv import load_dotenv
import os

# Load env variables
load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

key = os.getenv("key")
endpoint = os.getenv("endpoint")
deployment_name = os.getenv("DEPLOYMENT_NAME")

In [11]:
import requests
import time

In [12]:
# API URL
url = f"{endpoint}/formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

headers = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": key
}

data = {
    "urlSource": "https://raw.githubusercontent.com/Nainamitt/Cg-CapstoneProject-18April/main/InsuranceClaimImage.png"
}

response = requests.post(url, headers=headers, json=data)

print("Status Code:", response.status_code)

if response.status_code != 202:
    print("❌ ERROR:", response.text)
    raise Exception("Fix error before proceeding")

operation_url = response.headers.get("Operation-Location")

print("✅ Operation URL:", operation_url)

Status Code: 202
✅ Operation URL: https://capstone2345.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-document/analyzeResults/6394efb6-8657-4c82-989d-803674e174a6?api-version=2023-07-31


In [13]:
url = f"{endpoint}/formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

headers = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": key
}

data = {
    "urlSource": "https://raw.githubusercontent.com/Nainamitt/Cg-CapstoneProject-18April/main/InsuranceClaimImage.png"
}

response = requests.post(url, headers=headers, json=data)

# Get operation URL
operation_url = response.headers["Operation-Location"]

print("Request sent. Processing...")

Request sent. Processing...


In [14]:
while True:
    result = requests.get(operation_url, headers=headers)
    result_json = result.json()
    
    status = result_json["status"]
    print("Status:", status)
    
    if status == "succeeded":
        break
    elif status == "failed":
        raise Exception("Analysis failed")
    
    time.sleep(2)

print("✅ Extraction completed")

Status: succeeded
✅ Extraction completed


In [15]:
content = result_json["analyzeResult"]["content"]

print("📄 Raw Text:\n", content)

📄 Raw Text:
 SecureLife INSURANCE Your Trust, Our Promise
Claim No.
CLM/2025/000123
Date
10-02-2025
INSURANCE CLAIM FORM
Please fill in the form in BLOCK LETTERS and submit all required documents.
1. POLICYHOLDER DETAILS
Policy Number
POL12345
Policy Start Date
01-01-2024
Policyholder Name
Rahul Sharma
Policy End Date
31-12-2024
Date of Birth
15-05-1990
Email ID
rahul.sharma@email.com
Phone Number
9876543210
Address
123, MG Road, Connaught Place, New Delhi - 110001
2. CLAIM DETAILS
Claim Type :selected: Vehicle :unselected: Health :unselected: Property
Date of Claim
10-02-2025
Claim Amount (₹)
75,000
Place of Incident
Delhi
Previous Claims
2
Incident Time
14:30 PM
Description of Incident
Car met with an accident on highway near Akshardham. Front part severely damaged.
3. BANK DETAILS (FOR CLAIM SETTLEMENT)
Account Holder Name
Rahul Sharma
Bank Name
HDFC Bank
Account Number
50100212345678
IFSC Code
HDFC0001234
Branch
Connaught Place, New Delhi
Account Type
Savings
4. DOCUMENT CHECKLIST 

In [16]:
import re

def extract_fields(text):
    data = {}
    
    # Name
    name_match = re.search(r"Policyholder Name[:\s]*([A-Za-z ]+)", text)
    if name_match:
        data["Name"] = name_match.group(1)
    
    # Claim Amount
    amount_match = re.search(r"Claim Amount.*?(\d{2,6})", text)
    if amount_match:
        data["Claim_Amount"] = amount_match.group(1)
    
    # Date
    date_match = re.search(r"Date.*?(\d{2}-\d{2}-\d{4})", text)
    if date_match:
        data["Date"] = date_match.group(1)
    
    # Policy Number
    policy_match = re.search(r"Policy Number[:\s]*([A-Z0-9]+)", text)
    if policy_match:
        data["Policy_Number"] = policy_match.group(1)
    
    return data

In [17]:
final_data = extract_fields(content)
final_data

{'Name': 'Rahul Sharma', 'Date': '10-02-2025', 'Policy_Number': 'POL12345'}

In [18]:
# PART 2

In [19]:
# Azure AI Search

In [20]:
from dotenv import load_dotenv
import os

# Load env variables
load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

ADMIN_KEY = os.getenv("ADMIN_KEY")
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
INDEX_NAME = os.getenv("INDEX")

In [21]:
import requests

url = f"{SEARCH_ENDPOINT}/indexes?api-version=2021-04-30-Preview"

headers = {
    "api-key": ADMIN_KEY
}

response = requests.get(url, headers=headers)

print(response.status_code)
print(response.text)

200
{"@odata.context":"https://capstone.search.windows.net/$metadata#indexes","value":[{"@odata.etag":"\"0x8DE9D1008D8B0C9\"","name":"claims-index","defaultScoringProfile":null,"fields":[{"name":"id","type":"Edm.String","searchable":false,"filterable":false,"retrievable":true,"sortable":false,"facetable":false,"key":true,"indexAnalyzer":null,"searchAnalyzer":null,"analyzer":null,"normalizer":null,"synonymMaps":[]}],"scoringProfiles":[],"corsOptions":null,"suggesters":[],"analyzers":[],"normalizers":[],"tokenizers":[],"tokenFilters":[],"charFilters":[],"encryptionKey":null,"similarity":{"@odata.type":"#Microsoft.Azure.Search.BM25Similarity","k1":null,"b":null},"semantic":null}]}


In [22]:
import requests
import json

In [26]:
import requests

url = f"{SEARCH_ENDPOINT}/indexes/claims-index?api-version=2021-04-30-Preview"

headers = {
    "Content-Type": "application/json",
    "api-key": ADMIN_KEY
}

index_schema = {
    "name": "claims-index",
    "fields": [
        {"name": "id", "type": "Edm.String", "key": True},
        {"name": "Customer_ID", "type": "Edm.String"},
        {"name": "Claim_Amount", "type": "Edm.Int32", "filterable": True, "sortable": True},
        {"name": "Claim_Type", "type": "Edm.String", "searchable": True},
        {"name": "Location", "type": "Edm.String", "searchable": True, "filterable": True},
        {"name": "Previous_Claims", "type": "Edm.Int32", "filterable": True},
        {"name": "Fraud", "type": "Edm.String", "filterable": True},
        {"name": "content", "type": "Edm.String", "searchable": True}
    ]
}

response = requests.put(url, headers=headers, json=index_schema)
print(response.status_code)
print(response.text)

201
{"@odata.context":"https://capstone.search.windows.net/$metadata#indexes/$entity","@odata.etag":"\"0x8DE9D11083E5AD1\"","name":"claims-index","defaultScoringProfile":null,"fields":[{"name":"id","type":"Edm.String","searchable":true,"filterable":true,"retrievable":true,"sortable":true,"facetable":true,"key":true,"indexAnalyzer":null,"searchAnalyzer":null,"analyzer":null,"normalizer":null,"synonymMaps":[]},{"name":"Customer_ID","type":"Edm.String","searchable":true,"filterable":true,"retrievable":true,"sortable":true,"facetable":true,"key":false,"indexAnalyzer":null,"searchAnalyzer":null,"analyzer":null,"normalizer":null,"synonymMaps":[]},{"name":"Claim_Amount","type":"Edm.Int32","searchable":false,"filterable":true,"retrievable":true,"sortable":true,"facetable":true,"key":false,"indexAnalyzer":null,"searchAnalyzer":null,"analyzer":null,"normalizer":null,"synonymMaps":[]},{"name":"Claim_Type","type":"Edm.String","searchable":true,"filterable":true,"retrievable":true,"sortable":true,"

In [29]:
import pandas as pd

df = pd.read_csv("insurance_claims_large.csv")
df["id"] = df.index.astype(str)

docs = []

for _, row in df.iterrows():
    docs.append({
        "@search.action": "upload",
        "id": str(row["id"]),
        "Customer_ID": row["Customer_ID"],
        "Claim_Amount": int(row["Claim_Amount"]),
        "Claim_Type": row["Claim_Type"],
        "Location": row["Location"],
        "Previous_Claims": int(row["Previous_Claims"]),
        "Fraud": row["Fraud"],
        "content": f"{row['Claim_Type']} claim in {row['Location']}"
    })

upload_url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/index?api-version=2021-04-30-Preview"

response = requests.post(upload_url, headers=headers, json={"value": docs})

print(response.status_code)
print(response.text)

200
{"@odata.context":"https://capstone.search.windows.net/indexes('claims-index')/$metadata#Collection(Microsoft.Azure.Search.V2021_04_30_Preview.IndexResult)","value":[{"key":"0","status":true,"errorMessage":null,"statusCode":201},{"key":"1","status":true,"errorMessage":null,"statusCode":201},{"key":"2","status":true,"errorMessage":null,"statusCode":201},{"key":"3","status":true,"errorMessage":null,"statusCode":201},{"key":"4","status":true,"errorMessage":null,"statusCode":201},{"key":"5","status":true,"errorMessage":null,"statusCode":201},{"key":"6","status":true,"errorMessage":null,"statusCode":201},{"key":"7","status":true,"errorMessage":null,"statusCode":201},{"key":"8","status":true,"errorMessage":null,"statusCode":201},{"key":"9","status":true,"errorMessage":null,"statusCode":201},{"key":"10","status":true,"errorMessage":null,"statusCode":201},{"key":"11","status":true,"errorMessage":null,"statusCode":201},{"key":"12","status":true,"errorMessage":null,"statusCode":201},{"key":"

In [30]:
url = f"{SEARCH_ENDPOINT}/indexes/claims-index?api-version=2021-04-30-Preview"

res = requests.get(url, headers={"api-key": ADMIN_KEY})
print(res.json())

{'@odata.context': 'https://capstone.search.windows.net/$metadata#indexes/$entity', '@odata.etag': '"0x8DE9D11083E5AD1"', 'name': 'claims-index', 'defaultScoringProfile': None, 'fields': [{'name': 'id', 'type': 'Edm.String', 'searchable': True, 'filterable': True, 'retrievable': True, 'sortable': True, 'facetable': True, 'key': True, 'indexAnalyzer': None, 'searchAnalyzer': None, 'analyzer': None, 'normalizer': None, 'synonymMaps': []}, {'name': 'Customer_ID', 'type': 'Edm.String', 'searchable': True, 'filterable': True, 'retrievable': True, 'sortable': True, 'facetable': True, 'key': False, 'indexAnalyzer': None, 'searchAnalyzer': None, 'analyzer': None, 'normalizer': None, 'synonymMaps': []}, {'name': 'Claim_Amount', 'type': 'Edm.Int32', 'searchable': False, 'filterable': True, 'retrievable': True, 'sortable': True, 'facetable': True, 'key': False, 'indexAnalyzer': None, 'searchAnalyzer': None, 'analyzer': None, 'normalizer': None, 'synonymMaps': []}, {'name': 'Claim_Type', 'type': '

In [31]:
search_url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2021-04-30-Preview"

query = {
    "search": "fraud claim"
}

response = requests.post(search_url, headers=headers, json=query)
print(response.json())

{'@odata.context': "https://capstone.search.windows.net/indexes('claims-index')/$metadata#docs(*)", '@search.nextPageParameters': {'search': 'fraud claim', 'skip': 50}, 'value': [{'@search.score': 0.013605652, 'id': '29', 'Customer_ID': 'C0030', 'Claim_Amount': 160224, 'Claim_Type': 'Vehicle', 'Location': 'Pune', 'Previous_Claims': 0, 'Fraud': 'No', 'content': 'Vehicle claim in Pune'}, {'@search.score': 0.013605652, 'id': '38', 'Customer_ID': 'C0039', 'Claim_Amount': 162288, 'Claim_Type': 'Health', 'Location': 'Mumbai', 'Previous_Claims': 5, 'Fraud': 'Yes', 'content': 'Health claim in Mumbai'}, {'@search.score': 0.013605652, 'id': '73', 'Customer_ID': 'C0074', 'Claim_Amount': 180881, 'Claim_Type': 'Vehicle', 'Location': 'Chennai', 'Previous_Claims': 2, 'Fraud': 'Yes', 'content': 'Vehicle claim in Chennai'}, {'@search.score': 0.013605652, 'id': '91', 'Customer_ID': 'C0092', 'Claim_Amount': 28361, 'Claim_Type': 'Property', 'Location': 'Bangalore', 'Previous_Claims': 2, 'Fraud': 'No', 'co

In [32]:
search_url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2021-04-30-Preview"
query = {
    "search": "fraud",
    "filter": "Location eq 'Delhi' and Fraud eq 'Yes'"
}
response = requests.post(search_url, headers=headers, json=query)
print(response.json())

{'@odata.context': "https://capstone.search.windows.net/indexes('claims-index')/$metadata#docs(*)", 'value': []}


In [37]:
search_url = f"{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version=2021-04-30-Preview"

query = {
    "search": "*",
    "filter": "Claim_Amount gt 50000 and Location eq 'Delhi' and Fraud eq 'Yes'"
}
response = requests.post(search_url, headers=headers, json=query)
print(response.json())

{'@odata.context': "https://capstone.search.windows.net/indexes('claims-index')/$metadata#docs(*)", 'value': [{'@search.score': 1.0, 'id': '69', 'Customer_ID': 'C0070', 'Claim_Amount': 172733, 'Claim_Type': 'Health', 'Location': 'Delhi', 'Previous_Claims': 3, 'Fraud': 'Yes', 'content': 'Health claim in Delhi'}, {'@search.score': 1.0, 'id': '136', 'Customer_ID': 'C0137', 'Claim_Amount': 85794, 'Claim_Type': 'Vehicle', 'Location': 'Delhi', 'Previous_Claims': 3, 'Fraud': 'Yes', 'content': 'Vehicle claim in Delhi'}, {'@search.score': 1.0, 'id': '381', 'Customer_ID': 'C0382', 'Claim_Amount': 133600, 'Claim_Type': 'Health', 'Location': 'Delhi', 'Previous_Claims': 5, 'Fraud': 'Yes', 'content': 'Health claim in Delhi'}, {'@search.score': 1.0, 'id': '62', 'Customer_ID': 'C0063', 'Claim_Amount': 135657, 'Claim_Type': 'Vehicle', 'Location': 'Delhi', 'Previous_Claims': 5, 'Fraud': 'Yes', 'content': 'Vehicle claim in Delhi'}, {'@search.score': 1.0, 'id': '352', 'Customer_ID': 'C0353', 'Claim_Amount

In [39]:
query = {
    "search": "*",
    "filter": "Claim_Amount gt 50000 and Location eq 'Delhi' and Fraud eq 'Yes'",
    "top": 1
}

response = requests.post(search_url, headers=headers, json=query)
print(response.json())

{'@odata.context': "https://capstone.search.windows.net/indexes('claims-index')/$metadata#docs(*)", 'value': [{'@search.score': 1.0, 'id': '69', 'Customer_ID': 'C0070', 'Claim_Amount': 172733, 'Claim_Type': 'Health', 'Location': 'Delhi', 'Previous_Claims': 3, 'Fraud': 'Yes', 'content': 'Health claim in Delhi'}]}
